# YOUR PROJECT TITLE

> **Note the following:** 
> 1. This is *not* meant to be an example of an actual **data analysis project**, just an example of how to structure such a project.
> 1. Remember the general advice on structuring and commenting your code
> 1. The `dataproject.py` file includes a function which can be used multiple times in this notebook.

In [1]:
# The DST API wrapper
%pip install git+https://github.com/alemartinello/dstapi

  Cloning https://github.com/alemartinello/dstapi to /private/var/folders/lb/2stvshn97wz4m8c3hcpvz9bh0000gn/T/pip-req-build-gfhbczj8
  Running command git clone --filter=blob:none --quiet https://github.com/alemartinello/dstapi /private/var/folders/lb/2stvshn97wz4m8c3hcpvz9bh0000gn/T/pip-req-build-gfhbczj8
  Resolved https://github.com/alemartinello/dstapi to commit d9eeb5a82cbc70b7d63b2ff44d92632fd77123a4
  Preparing metadata (setup.py) ... done
Note: you may need to restart the kernel to use updated packages.


Imports and set magics:

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
#from matplotlib_venn import venn2
import re

# autoreload modules when code is run
%load_ext autoreload
%autoreload 2

# user written modules
import dataproject

from dstapi import DstApi


# Read and clean data

Importing data, through an API and loading it:

In [3]:
proj=DstApi('FRKM123')

A quick overview of the available data

In [4]:
tabsum = proj.tablesummary(language='en')
display(tabsum)

Table FRKM123: Population projections 2023 by region, age, sex and time
Last update: 2023-06-01T08:00:00


,variable name,# values,First value,First value label,Last value,Last value label,Time variable
0,OMRÅDE,99,101,Copenhagen,851,Aalborg,False
1,ALDER,102,TOT,"Age, total",100-,100 years and over,False
2,KØN,2,M,Men,K,Women,False
3,Tid,28,2023,2023,2050,2050,True


***Clean data***

In [5]:
params = proj._define_base_params(language='en')
params

{'table': 'frkm123',
 'format': 'BULK',
 'lang': 'en',
 'variables': [{'code': 'OMRÅDE', 'values': ['*']},
  {'code': 'ALDER', 'values': ['*']},
  {'code': 'KØN', 'values': ['*']},
  {'code': 'Tid', 'values': ['*']}]}

In [6]:
proj_api=proj.get_data(params=params)
proj_api.head(5)

,OMRÅDE,ALDER,KØN,TID,INDHOLD
0,Roskilde,15 years,Women,2036,529
1,Roskilde,15 years,Men,2036,588
2,Roskilde,16 years,Women,2036,553
3,Roskilde,16 years,Men,2036,558
4,Roskilde,17 years,Women,2036,552


Rename variables to English:

In [7]:
proj_api.rename(columns = {'OMRÅDE':'municipality'}, inplace=True)
proj_api.rename(columns = {'ALDER':'age'}, inplace=True)
proj_api.rename(columns = {'KØN':'sex'}, inplace=True)
proj_api.rename(columns = {'TID':'year'}, inplace=True)
proj_api.rename(columns = {'INDHOLD':'population_projection'}, inplace=True)
proj_api.head(5)

,municipality,age,sex,year,population_projection
0,Roskilde,15 years,Women,2036,529
1,Roskilde,15 years,Men,2036,588
2,Roskilde,16 years,Women,2036,553
3,Roskilde,16 years,Men,2036,558
4,Roskilde,17 years,Women,2036,552


Turn the data into a dataframe and add a new column that summarizes the population projection of men and women for the given municipality, age and year:

In [8]:
df=pd.DataFrame(proj_api)
df['pop_proj_all'] = ""

df.head(5)

,municipality,age,sex,year,population_projection,pop_proj_all
0,Roskilde,15 years,Women,2036,529,
1,Roskilde,15 years,Men,2036,588,
2,Roskilde,16 years,Women,2036,553,
3,Roskilde,16 years,Men,2036,558,
4,Roskilde,17 years,Women,2036,552,


We remove the municipality Christiansø as the population value is 0 for all years:

In [9]:
df=df[df['municipality']!='Christiansø']

In [10]:
#Summarize values of population_projection if municipality equals each other, age equals each other and year equals each other:
def sumval(group):
    if len(group)>1:
        return group.sum()
    else:
        return group.values[0]
    
df['pop_proj_all']=df.groupby(['municipality','age','year'])['population_projection'].transform(sumval)
df.head(5)

,municipality,age,sex,year,population_projection,pop_proj_all
0,Roskilde,15 years,Women,2036,529,1117
1,Roskilde,15 years,Men,2036,588,1117
2,Roskilde,16 years,Women,2036,553,1111
3,Roskilde,16 years,Men,2036,558,1111
4,Roskilde,17 years,Women,2036,552,1133


Deleting half of the observations. We delete the rows where sex = Men such that we only have one row for every municipality, age and sex.

In [11]:
df=df[df['sex']!='Men']
df.head(5)

,municipality,age,sex,year,population_projection,pop_proj_all
0,Roskilde,15 years,Women,2036,529,1117
2,Roskilde,16 years,Women,2036,553,1111
4,Roskilde,17 years,Women,2036,552,1133
6,Roskilde,18 years,Women,2036,593,1148
8,Roskilde,19 years,Women,2036,582,1183


In [12]:
columns_to_drop=['sex','population_projection']
df=df.drop(columns_to_drop, axis=1)

In [13]:
df.head(5)

,municipality,age,year,pop_proj_all
0,Roskilde,15 years,2036,1117
2,Roskilde,16 years,2036,1111
4,Roskilde,17 years,2036,1133
6,Roskilde,18 years,2036,1148
8,Roskilde,19 years,2036,1183


In [14]:
muni_type=df['municipality'].dtype
age_type=df['age'].dtype
year_type=df['year'].dtype
pop_type=df['pop_proj_all'].dtype
print(muni_type)
print(age_type)
print(year_type)
print(pop_type)

object
object
int64
int64


In [15]:
print(type('age'))

<class 'str'>


In [16]:
df=df[df['age']!='Age, total']

In [17]:
df['age'] = df['age'].str.split(" y").str[0].astype(int)
df.head(5)

,municipality,age,year,pop_proj_all
0,Roskilde,15,2036,1117
2,Roskilde,16,2036,1111
4,Roskilde,17,2036,1133
6,Roskilde,18,2036,1148
8,Roskilde,19,2036,1183


In [18]:
# Define age ranges
age_bins = range(df['age'].min(), df['age'].max() + 11, 10)

# Create labels for the age ranges
labels = [f'{i}-{i+9}' if i != 100 else f'{i}+' for i in age_bins[:-1]]

# Use pd.cut() to sort ages into age ranges
df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=labels, right=False)

# Function to summarize values in 'Value' column for each group
def sumval(group):
    if len(group) > 1:
        return group.sum()
    else:
        return group.values[0]

# Applying the function to create the new column
df['pop_proj_age'] = df.groupby(['municipality', 'age_group', 'year'])['pop_proj_all'].transform(sumval)

print(df)

/var/folders/lb/2stvshn97wz4m8c3hcpvz9bh0000gn/T/ipykernel_87042/887293629.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df['pop_proj_age'] = df.groupby(['municipality', 'age_group', 'year'])['pop_proj_all'].transform(sumval)


       municipality  age  year  pop_proj_all age_group  pop_proj_age
0          Roskilde   15  2036          1117     10-19         11095
2          Roskilde   16  2036          1111     10-19         11095
4          Roskilde   17  2036          1133     10-19         11095
6          Roskilde   18  2036          1148     10-19         11095
8          Roskilde   19  2036          1183     10-19         11095
...             ...  ...   ...           ...       ...           ...
565478   Jammerbugt   42  2050           418     40-49          4248
565480   Jammerbugt   43  2050           429     40-49          4248
565482   Jammerbugt   44  2050           425     40-49          4248
565484   Jammerbugt   45  2050           427     40-49          4248
565486   Jammerbugt   46  2050           427     40-49          4248

[277144 rows x 6 columns]


In [19]:
df = df.sort_values(by=['municipality','year','age'])
df.head(5)

,municipality,age,year,pop_proj_all,age_group,pop_proj_age
140938,Aabenraa,0,2023,501,0-9,5820
140940,Aabenraa,1,2023,547,0-9,5820
140964,Aabenraa,2,2023,540,0-9,5820
140986,Aabenraa,3,2023,592,0-9,5820
141008,Aabenraa,4,2023,637,0-9,5820


In [20]:
df_unique = df.drop_duplicates(subset=['municipality', 'year', 'pop_proj_age'])
df_unique.head(5)

,municipality,age,year,pop_proj_all,age_group,pop_proj_age
140938,Aabenraa,0,2023,501,0-9,5820
140942,Aabenraa,10,2023,623,10-19,7129
140966,Aabenraa,20,2023,677,20-29,5297
140988,Aabenraa,30,2023,629,30-39,6192
141010,Aabenraa,40,2023,605,40-49,7020


In [21]:
columns_to_drop_2=['age','pop_proj_all']
df=df_unique.drop(columns_to_drop_2, axis=1)


In [22]:
df.head(5)

,municipality,year,age_group,pop_proj_age
140938,Aabenraa,2023,0-9,5820
140942,Aabenraa,2023,10-19,7129
140966,Aabenraa,2023,20-29,5297
140988,Aabenraa,2023,30-39,6192
141010,Aabenraa,2023,40-49,7020


In [23]:
#Resetting the index:
df.reset_index(inplace = True, drop = True) # Drop old index too
df.rename(columns = {'pop_proj_age':'population_projection'}, inplace=True)
df.head(5)

,municipality,year,age_group,population_projection
0,Aabenraa,2023,0-9,5820
1,Aabenraa,2023,10-19,7129
2,Aabenraa,2023,20-29,5297
3,Aabenraa,2023,30-39,6192
4,Aabenraa,2023,40-49,7020


Adding Denmark in total to the dataframe. We do it for every combination of year and age group:

In [24]:
# Calculate the sum of population_projection for each combination of 'year' and 'age_group':
summarized_data = df.groupby(['year', 'age_group'])['population_projection'].sum().reset_index()

# Concatenate the original DataFrame with the summarized data, adding the sums of population_projection for each 'year' and 'age_group' as new rows:
df = pd.concat([df, summarized_data], ignore_index=True)

# Determine the index where the new rows start, which represents the original length of the DataFrame minus the number of rows added (length of summarized_data):
original_length = len(df) - len(summarized_data)

# Set the municipality for the newly added rows to 'Denmark':
df.loc[original_length:, 'municipality'] = 'Denmark'

# Select and print only the new rows that were added to the DataFrame:
new_rows = df.iloc[original_length:]
print(new_rows)

      municipality  year age_group  population_projection
30149      Denmark  2023       0-9                 617572
30150      Denmark  2023     10-19                 677784
30151      Denmark  2023     20-29                 784929
30152      Denmark  2023     30-39                 740744
30153      Denmark  2023     40-49                 727672
...            ...   ...       ...                    ...
30452      Denmark  2050     60-69                 632702
30453      Denmark  2050     70-79                 642350
30454      Denmark  2050     80-89                 487132
30455      Denmark  2050     90-99                 121395
30456      Denmark  2050      100+                   4249

[308 rows x 4 columns]


/var/folders/lb/2stvshn97wz4m8c3hcpvz9bh0000gn/T/ipykernel_87042/33886451.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summarized_data = df.groupby(['year', 'age_group'])['population_projection'].sum().reset_index()


Summarizing the projected population for each municipality for each year:

In [25]:
#creating a group by municipality and year
summed_population = df.groupby(['municipality', 'year'])['population_projection'].sum().reset_index()
total_population = summed_population.copy()
#calling the total projected population 'sum of ages'
total_population['age_group'] = 'sum of ages'
#calculating the sum for each group
total_population['population_projection'] = summed_population.groupby(['municipality', 'year'])['population_projection'].transform('sum')
#concatenating the total population onto the existing dataframe
df = pd.concat([df, total_population], ignore_index=True)

df = df.sort_values(by=['municipality','year'])
print(df)

      municipality  year    age_group  population_projection
0         Aabenraa  2023          0-9                   5820
1         Aabenraa  2023        10-19                   7129
2         Aabenraa  2023        20-29                   5297
3         Aabenraa  2023        30-39                   6192
4         Aabenraa  2023        40-49                   7020
...            ...   ...          ...                    ...
30145          Ærø  2050        70-79                    939
30146          Ærø  2050        80-89                    780
30147          Ærø  2050        90-99                    211
30148          Ærø  2050         100+                      8
33228          Ærø  2050  sum of ages                   5848

[33229 rows x 4 columns]


In [26]:
#creating a new column 'pop_share' to find the population share for each age group
df = df.merge(total_population[['municipality', 'year', 'population_projection']], on=['municipality', 'year'], suffixes=('', '_total'))
df['pop_share'] = df['population_projection'] / df['population_projection_total']
df=df.drop('population_projection_total', axis=1)
print(df)

      municipality  year    age_group  population_projection  pop_share
0         Aabenraa  2023          0-9                   5820   0.098641
1         Aabenraa  2023        10-19                   7129   0.120826
2         Aabenraa  2023        20-29                   5297   0.089777
3         Aabenraa  2023        30-39                   6192   0.104946
4         Aabenraa  2023        40-49                   7020   0.118979
...            ...   ...          ...                    ...        ...
33224          Ærø  2050        70-79                    939   0.160568
33225          Ærø  2050        80-89                    780   0.133379
33226          Ærø  2050        90-99                    211   0.036081
33227          Ærø  2050         100+                      8   0.001368
33228          Ærø  2050  sum of ages                   5848   1.000000

[33229 rows x 5 columns]


# Analysis

***SUMMARY STATISTICS***

In [44]:
# Get the number of unique municipalities, years, and age groups
num_years = df['year'].nunique()
num_age_group_2023 = df[(df['year'] == 2023) & (df['age_group'] != 'sum of ages')]['age_group'].nunique()
num_municipalities = df[(df['year'] == 2023) & (df['municipality'] != 'Denmark')]['municipality'].nunique()

#Create the min and max years:
min_year = df['year'].min()
max_year = df['year'].max()

# Filter the DataFrame for the year 2023, age group 'sum of ages', and excluding the municipality 'Denmark'
filtered_data = df[(df['year'] == 2023) & (df['age_group'] == 'sum of ages') & (df['municipality'] != 'Denmark')]

# Calculate the descriptive statistics for the population projection
projection_stats = filtered_data['population_projection'].describe()
projection_median = filtered_data['population_projection'].median()



descriptive_table = pd.DataFrame({
    'Attribute': ['Number of Municipalities', 'Number of Years', 'First Year', 'Last Year', 'Number of Age Groups',
                   'Min Population Projection (2023)', 'Max Population Projection (2023)',
                   'Mean Population Projection (2023)', 'Median Population Projection (2023)'],
    'Value': [num_municipalities, num_years, min_year, max_year, num_age_group_2023,
              projection_stats['min'], projection_stats['max'], 
              projection_stats['mean'], projection_median]
})

# Convert scientific notation to normal numbers
descriptive_table['Value'] = descriptive_table['Value'].apply(lambda x: '{:,.0f}'.format(x) if isinstance(x, (int, float)) else x)

# Print the descriptive table
descriptive_table.head(10)

,Attribute,Value
0,Number of Municipalities,98
1,Number of Years,28
2,First Year,"2,023"
3,Last Year,"2,050"
4,Number of Age Groups,11
5,Min Population Projection (2023),"1,789"
6,Max Population Projection (2023),"653,664"
7,Mean Population Projection (2023),"60,533"
8,Median Population Projection (2023),"43,795"


We want to find the municipalities with the minimum and maximum value of population:

In [31]:
# Find the index of the row where the minimum population projection for the year 2023 occurs
min_projection_index = filtered_data[filtered_data['population_projection'] == projection_stats['min']].index[0]

# Retrieve the entire row corresponding to the minimum population projection
min_projection_row = filtered_data.loc[min_projection_index]

print("Row with the minimum population projection for 2023:")
print(min_projection_row)

# Find the index of the row where the maximum population projection for the year 2023 occurs
max_projection_index = filtered_data[filtered_data['population_projection'] == projection_stats['max']].index[0]

# Retrieve the entire row corresponding to the minimum population projection
max_projection_row = filtered_data.loc[max_projection_index]

print("Row with the maximum population projection for 2023:")
print(max_projection_row)

Row with the minimum population projection for 2023:
municipality                    Læsø
year                            2023
age_group                sum of ages
population_projection           1789
pop_share                        1.0
Name: 19485, dtype: object
Row with the maximum population projection for 2023:
municipality              Copenhagen
year                            2023
age_group                sum of ages
population_projection         653664
pop_share                        1.0
Name: 3701, dtype: object


MAKE FURTHER ANALYSIS. EXPLAIN THE CODE BRIEFLY AND SUMMARIZE THE RESULTS.

# Conclusion

ADD CONCISE CONLUSION.